# Ансамбли моделей машинного обучения. Часть 2.

In [2]:
import pandas as pd

### 1. Выберите набор данных (датасет) для решения задачи классификации или регресии.

Используем данные из Healthy Diet & Calorie Intake: Nutrition & Health https://www.kaggle.com/datasets/aliyasaly1231/healthy-diet-and-calorie-intake

In [3]:
data = pd.read_csv('healthy_diet_calorie_intake.csv', sep=",")

In [4]:
# размер набора данных
data.shape

(6000, 15)

In [5]:
# типы колонок
data.dtypes

Person_ID                        str
Age                            int64
Gender                           str
Height_cm                    float64
Weight_kg                    float64
BMI                          float64
Activity_Level                   str
Daily_Calorie_Requirement      int64
Daily_Calorie_Consumed         int64
Protein_Intake_g             float64
Carbohydrate_Intake_g        float64
Fat_Intake_g                 float64
Water_Intake_Liters          float64
Diet_Type                        str
Health_Status                    str
dtype: object

### 2. В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.

In [6]:
data.isnull().sum()

Person_ID                    0
Age                          0
Gender                       0
Height_cm                    0
Weight_kg                    0
BMI                          0
Activity_Level               0
Daily_Calorie_Requirement    0
Daily_Calorie_Consumed       0
Protein_Intake_g             0
Carbohydrate_Intake_g        0
Fat_Intake_g                 0
Water_Intake_Liters          0
Diet_Type                    0
Health_Status                0
dtype: int64

In [7]:
data.head()

,Person_ID,Age,Gender,Height_cm,Weight_kg,BMI,Activity_Level,Daily_Calorie_Requirement,Daily_Calorie_Consumed,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,Water_Intake_Liters,Diet_Type,Health_Status
0,P0001,50,Male,176.4,74.8,24.0,Very Active,2852,2625,183.0,16.9,202.8,3.3,Keto,Healthy
1,P0002,18,Female,167.6,75.5,26.9,Sedentary,1904,2044,90.1,306.5,50.8,1.9,Vegan,Overweight
2,P0003,68,Female,161.9,87.2,33.3,Lightly Active,2009,2540,222.7,281.3,58.2,2.4,High Protein,Obese
3,P0004,22,Female,169.3,66.9,23.3,Moderately Active,2318,2096,69.5,299.8,68.7,2.9,Balanced,Healthy
4,P0005,30,Male,179.1,75.3,23.5,Sedentary,2144,1937,32.9,285.6,73.7,2.2,Balanced,Healthy


Я выбрал регрессию по weight_kg но у нас есть категориальные признаки, закодирую их с LabelEncoding

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data = data.drop('Person_ID', axis=1)

data['Gender'] = le.fit_transform(data['Gender'])
data['Activity_Level'] = le.fit_transform(data['Activity_Level'])
data['Diet_Type'] = le.fit_transform(data['Diet_Type'])
data['Health_Status'] = le.fit_transform(data['Health_Status'])

In [9]:
data.head()

,Age,Gender,Height_cm,Weight_kg,BMI,Activity_Level,Daily_Calorie_Requirement,Daily_Calorie_Consumed,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,Water_Intake_Liters,Diet_Type,Health_Status
0,50,1,176.4,74.8,24.0,4,2852,2625,183.0,16.9,202.8,3.3,2,0
1,18,0,167.6,75.5,26.9,3,1904,2044,90.1,306.5,50.8,1.9,4,2
2,68,0,161.9,87.2,33.3,1,2009,2540,222.7,281.3,58.2,2.4,1,1
3,22,0,169.3,66.9,23.3,2,2318,2096,69.5,299.8,68.7,2.9,0,0
4,30,1,179.1,75.3,23.5,3,2144,1937,32.9,285.6,73.7,2.2,0,0


### 3. С использованием метода train_test_split разделите выборку на обучающую и тестовую.

In [10]:
X = data.drop('Weight_kg', axis=1)
y = data['Weight_kg']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)
print("Размер y_train:", y_train.shape)
print("Размер y_test:", y_test.shape)

Размер обучающей выборки: (3600, 13)
Размер тестовой выборки: (2400, 13)
Размер y_train: (3600,)
Размер y_test: (2400,)


### 4. Обучите следующие ансамблевые модели

#### Одну из моделей группы стекинга.

Стекинг (Stacking) — ансамблевый метод машинного обучения, в котором используется несколько моделей первого уровня и отдельная модель второго уровня (мета-ученик). Модели первого уровня обучаются независимо друг от друга и формируют прогнозы. Полученные прогнозы используются как входные признаки для модели второго уровня, которая строит окончательное предсказание. В данной работе в качестве моделей первого уровня использовались линейная регрессия, дерево решений и случайный лес, а в качестве мета-ученика — линейная регрессия. Такой подход позволяет объединить преимущества различных алгоритмов и повысить качество прогнозирования.

In [11]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [12]:
#создание стекинга
base_models = [
    ('lr', LinearRegression()),
    ('tree', DecisionTreeRegressor(random_state=42)),
    ('rf', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
]

stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression()
)

#обучение
stacking.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('lr', ...), ('tree', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",LinearRegression()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model

In [13]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

y_pred_stack = stacking.predict(X_test)

print("Stacking Regressor")
print("MAE =", mean_absolute_error(y_test, y_pred_stack))
print("MSE =", mean_squared_error(y_test, y_pred_stack))
print("R² =", r2_score(y_test, y_pred_stack))

Stacking Regressor
MAE = 0.4204761403161985
MSE = 0.4628842030883345
R² = 0.9969663337507328


Качество отдельных моделей

In [14]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print(name)
    print("R² =", r2_score(y_test, pred))
    print()

Linear Regression
R² = 0.9917098224318054

Decision Tree
R² = 0.988430598682623

Random Forest
R² = 0.9954427055375525



#### Модель многослойного персептрона

Многослойный персептрон (Multi-Layer Perceptron, MLP) представляет собой искусственную нейронную сеть прямого распространения сигнала. Сеть состоит из входного слоя, одного или нескольких скрытых слоев и выходного слоя. Каждый нейрон скрытого слоя выполняет взвешенное суммирование входных сигналов и применяет функцию активации.

Обучение сети осуществляется методом обратного распространения ошибки (Backpropagation), при котором веса нейронов корректируются для минимизации функции потерь.

В задаче регрессии используется класс MLPRegressor, который прогнозирует непрерывные значения. Для повышения эффективности обучения входные признаки предварительно масштабируются с помощью стандартизации (StandardScaler), поскольку многослойный персептрон чувствителен к масштабу данных.

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(
    hidden_layer_sizes=(100, 50),  # два скрытых слоя
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [18]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

y_pred = mlp.predict(X_test_scaled)

print("MLP Regressor")
print("MAE =", mean_absolute_error(y_test, y_pred))
print("MSE =", mean_squared_error(y_test, y_pred))
print("R² =", r2_score(y_test, y_pred))

MLP Regressor
MAE = 0.2712133984881581
MSE = 0.14118349307828357
R² = 0.9990747068164184


#### (дополнительно) двумя методами на выбор из семейства МГУА

gmdh не работает на новом питоне

In [39]:
from gmdh import Combi, Mia
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ModuleNotFoundError: No module named 'gmdh'

In [ ]:
combi = Combi()

combi.fit(X_train, y_train)

y_pred_combi = combi.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_combi)
mse = mean_squared_error(y_test, y_pred_combi)
r2 = r2_score(y_test, y_pred_combi)

print("COMBI")
print("MAE =", mae)
print("MSE =", mse)
print("R2 =", r2)

In [30]:
mia = Mia()

mia.fit(X_train, y_train)

y_pred_mia = mia.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_mia)
mse = mean_squared_error(y_test, y_pred_mia)
r2 = r2_score(y_test, y_pred_mia)

print("MIA")
print("MAE =", mae)
print("MSE =", mse)
print("R2 =", r2)

NameError: name 'Mia' is not defined

### 5. Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.

In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append([name, mae, mse, r2])

import pandas as pd

df_results = pd.DataFrame(results, columns=["Model", "MAE", "MSE", "R2"])
df_results = df_results.sort_values(by="MAE")

df_results

,Model,MAE,MSE,R2
2,Random Forest,0.408259,0.695363,0.995443
0,Linear Regression,0.782914,1.264936,0.991710
1,Decision Tree,0.880292,1.765287,0.988431


В ходе эксперимента были обучены три регрессионные модели:
линейная регрессия, дерево решений и случайный лес. Для оценки качества использовались метрики MAE, MSE и R².

Наилучшие результаты показала модель Random Forest, которая имеет:

- минимальную ошибку MAE = 0.408
- минимальную ошибку MSE = 0.695
- максимальное значение коэффициента детерминации R² = 0.995

Это означает, что модель объясняет около 99.5% вариации целевой переменной, что свидетельствует о очень высокой точности предсказаний.


Можно сделать вывод, что:

- ансамблевые методы (особенно Random Forest) обеспечивают наилучшую точность;
- линейные модели дают стабильный, но менее точный результат;
- одиночное дерево решений уступает более сложным моделям.